# PART 03 · 5장 작업형 2 제출 파일 생성

교재 279~298쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 01 result.csv 생성

### 1 제출 파일의 역할 이해

### 2 result.csv 만들기

#### 1. 데이터 준비

이진분류 데이터를 불러와 전처리까지 준비해둔다.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
binary_base = 'data/part03_type2/binary_learning_renewal'
binary_train = pd.read_csv(binary_base+'/train.csv')
binary_test = pd.read_csv(binary_base+'/test.csv')
binary_submit = pd.read_csv(binary_base+'/sample_submission.csv')
binary_target = 'renew_next_month'
binary_X = binary_train.drop(
    columns=[binary_target, 'student_id']
).copy()
binary_y = binary_train[binary_target].copy()
binary_test_X = binary_test.drop(columns=['student_id']).copy()
num_cols = binary_X.select_dtypes(include='number').columns
cat_cols = binary_X.select_dtypes(exclude='number').columns
for col in num_cols:
    fill_value = binary_X[col].median()
    binary_X[col] = binary_X[col].fillna(fill_value)
    binary_test_X[col] = binary_test_X[col].fillna(fill_value)
for col in cat_cols:
    fill_value = binary_X[col].mode()[0]
    binary_X[col] = binary_X[col].fillna(fill_value)
    binary_test_X[col] = binary_test_X[col].fillna(fill_value)
binary_X = pd.get_dummies(binary_X)
binary_test_X = pd.get_dummies(binary_test_X)
binary_test_X = binary_test_X.reindex(
    columns=binary_X.columns,
    fill_value=0
)
print(binary_train.shape)
print(binary_test.shape)
print(binary_submit.shape)
print(binary_X.shape, binary_test_X.shape)

(720, 15)
(240, 14)
(240, 1)
(720, 28) (240, 28)


#### 2. 전체 train 학습과 result.csv 저장

전체 train으로 학습한 모델로 test 확률을 만들어 result.csv로 저장해본다.

In [2]:
binary_model = RandomForestClassifier(
    n_estimators=100,
    random_state=0
)
binary_model.fit(binary_X, binary_y)
positive_index = list(binary_model.classes_).index(1)
binary_pred = binary_model.predict_proba(binary_test_X)[:, positive_index]
binary_result = binary_submit.copy()
binary_result['pred'] = binary_pred
binary_result.to_csv('result.csv', index=False)
binary_check = pd.read_csv('result.csv')
print(binary_model.classes_)
print(binary_check.shape)
print(binary_check.head())
print(binary_check.columns.tolist())

[0 1]
(240, 1)
   pred
0  0.27
1  0.32
2  0.55
3  0.25
4  0.13
['pred']


#### 3. sample_submission을 복사하는 이유

## 02 제출 컬럼명 확인

### 1 제출 컬럼명 확인 방법

#### 1. 문제 문장과 sample_submission 확인

sample_submission의 크기, 컬럼명, 값의 형태를 확인해본다.

In [3]:
print(binary_submit.shape)
print(binary_submit.columns.tolist())
print(binary_submit.head())

(240, 1)
['pred']
   pred
0   0.0
1   0.0
2   0.0
3   0.0
4   0.0


#### 2. target 컬럼명과 제출 컬럼명 구분

### 2 저장 결과 비교와 오류 예시

#### 1. 저장한 파일과 제출 예시 비교

저장한 파일의 컬럼이 제출 예시와 같은지 코드로 비교해본다.

In [4]:
print(
    binary_check.columns.tolist()
     == binary_submit.columns.tolist()
)
missing_columns = [
    col for col in binary_submit.columns
    if col not in binary_check.columns
]
extra_columns = [
    col for col in binary_check.columns
    if col not in binary_submit.columns
]
print(missing_columns)
print(extra_columns)

True
[]
[]


#### 2. 잘못된 컬럼명 예시

## 03 index=False 저장

### 1 저장 방식 비교

#### 1. 두 저장 방식 비교

index=False를 지정했을 때와 생략했을 때의 결과를 비교해본다.

In [5]:
binary_result.to_csv(
    'result_without_index.csv',
    index=False
)
binary_result.to_csv(
    'result_with_index.csv'
)
without_index = pd.read_csv('result_without_index.csv')
with_index = pd.read_csv('result_with_index.csv')
print(without_index.shape)
print(without_index.columns.tolist())
print(with_index.shape)
print(with_index.columns.tolist())

(240, 1)
['pred']
(240, 2)
['Unnamed: 0', 'pred']


### 2 인덱스 컬럼 점검

#### 1. 불필요한 인덱스 컬럼 검사

Unnamed로 시작하는 컬럼이 있는지 검사하는 코드는 다음과 같다.

In [6]:
print(
    any(
        str(col).startswith('Unnamed')
        for col in without_index.columns
    )
)
print(
    any(
        str(col).startswith('Unnamed')
        for col in with_index.columns
    )
)

False
True


#### 2. 저장 전 인덱스를 삭제할 필요는 없음

## 04 확률값 제출 여부 확인

### 1 두 예측 방식 비교

#### 1. predict( )와 predict_proba( ) 비교

#### 2. 검증 데이터에서 두 예측값 비교

같은 모델에서 확률과 클래스값을 각각 만들어 형태를 비교해본다.

In [7]:
probability_model = binary_model
test_proba_all = probability_model.predict_proba(binary_test_X)
test_class = probability_model.predict(binary_test_X)
positive_class = 1
positive_index = list(probability_model.classes_).index(positive_class)
test_proba = test_proba_all[:, positive_index]
print(probability_model.classes_)
print(test_proba_all.shape)
print(test_class.shape)
print(test_proba.shape)

[0 1]
(240, 2)
(240,)
(240,)


### 2 확률 열 확인

#### 1. classes_로 확률 열 확인

predict_proba( )의 열 순서는 model.classes_의 순서와 같다. 이번에는 classes_가 [0 1]이므로 [:, 1]이 1 클래스 확률이다. 클래스값이 문자열이거나 순서가 다를 수 있으므로 관심 클래스의 위치를 확인하는 습관이 안전하다.

In [8]:
positive_class = 1
positive_index = list(binary_model.classes_).index(positive_class)
test_probability = binary_model.predict_proba(
    binary_test_X
)[:, positive_index]
print(test_probability.shape)
print(round(test_probability.min(), 2))
print(round(test_probability.max(), 2))

(240,)
0.08
0.94


## 05 클래스값 제출 여부 확인

### 1 클래스값 예측 확인

#### 1. 다중분류 데이터 준비

다중분류 데이터를 불러와 입력을 준비한다.

In [9]:
multi_base = 'data/part03_type2/multiclass_ticket_priority'
multi_train = pd.read_csv(multi_base+'/train.csv')
multi_test = pd.read_csv(multi_base+'/test.csv')
multi_submit = pd.read_csv(
    multi_base + '/sample_submission.csv'
)
multi_target = 'priority_level'
multi_X = multi_train.drop(
    columns=[multi_target, 'ticket_id']
).copy()
multi_y = multi_train[multi_target].copy()
multi_test_X = multi_test.drop(columns=['ticket_id']).copy()
num_cols = multi_X.select_dtypes(include='number').columns
cat_cols = multi_X.select_dtypes(exclude='number').columns
for col in num_cols:
    fill_value = multi_X[col].median()
    multi_X[col] = multi_X[col].fillna(fill_value)
    multi_test_X[col] = multi_test_X[col].fillna(fill_value)
for col in cat_cols:
    fill_value = multi_X[col].mode()[0]
    multi_X[col] = multi_X[col].fillna(fill_value)
    multi_test_X[col] = multi_test_X[col].fillna(fill_value)
multi_X = pd.get_dummies(multi_X)
multi_test_X = pd.get_dummies(multi_test_X)
multi_test_X = multi_test_X.reindex(
    columns=multi_X.columns,
    fill_value=0
)
print(multi_train.shape)
print(multi_test.shape)
print(multi_submit.shape)

(780, 15)
(260, 14)
(260, 1)


#### 2. predict( )와 predict_proba( ) 확인

학습한 모델에서 클래스값과 확률 배열의 형태를 확인해본다.

In [10]:
multi_model = RandomForestClassifier(
    n_estimators=100,
    random_state=0
)
multi_model.fit(multi_X, multi_y)
multi_test_class = multi_model.predict(multi_test_X)
multi_test_proba = multi_model.predict_proba(multi_test_X)
print(multi_model.classes_)
print(multi_test_class.shape)
print(multi_test_proba.shape)

[0 1 2]
(260,)
(260, 3)


### 2 클래스값 저장

#### 1. 전체 train으로 다시 학습하고 클래스값 저장

전체 train으로 학습한 클래스값을 result 파일로 저장하고 다시 읽어 확인해본다.

In [11]:
multi_model.fit(multi_X, multi_y)
multi_pred = multi_model.predict(multi_test_X)
multi_result = multi_submit.copy()
multi_result['pred'] = multi_pred
multi_result.to_csv(
    'result_multiclass.csv',
    index=False
)
multi_check = pd.read_csv('result_multiclass.csv')
print(multi_check.shape)
print(multi_check.head())
print(multi_check['pred'].value_counts().sort_index())
print(
    set(multi_check['pred'].unique()).issubset(
        set(multi_model.classes_)
    )
)

(260, 1)
   pred
0     0
1     0
2     1
3     1
4     1
pred
0    103
1    152
2      5
Name: count, dtype: int64
True


## 06 회귀 예측값 제출 확인

### 1 회귀 제출 준비

#### 1. 회귀 데이터 준비

회귀 데이터를 불러와 입력을 준비한다.

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
reg_base = 'data/part03_type2/regression_delivery_minutes'
reg_train = pd.read_csv(reg_base+'/train.csv')
reg_test = pd.read_csv(reg_base+'/test.csv')
reg_submit = pd.read_csv(
    reg_base + '/sample_submission.csv'
)
reg_target = 'delivery_minutes'
reg_X = reg_train.drop(
    columns=[reg_target, 'order_id']
).copy()
reg_y = reg_train[reg_target].copy()
reg_test_X = reg_test.drop(columns=['order_id']).copy()
num_cols = reg_X.select_dtypes(include='number').columns
cat_cols = reg_X.select_dtypes(exclude='number').columns
for col in num_cols:
    fill_value = reg_X[col].median()
    reg_X[col] = reg_X[col].fillna(fill_value)
    reg_test_X[col] = reg_test_X[col].fillna(fill_value)
for col in cat_cols:
    fill_value = reg_X[col].mode()[0]
    reg_X[col] = reg_X[col].fillna(fill_value)
    reg_test_X[col] = reg_test_X[col].fillna(fill_value)
reg_X = pd.get_dummies(reg_X)
reg_test_X = pd.get_dummies(reg_test_X)
reg_test_X = reg_test_X.reindex(
    columns=reg_X.columns,
    fill_value=0
)
print(reg_train.shape)
print(reg_test.shape)
print(reg_submit.shape)

(840, 15)
(280, 14)
(280, 1)


### 2 검증과 예측값 저장

#### 1. 검증 지표와 제출값 구분

검증 지표 계산과 제출값 생성은 사용하는 데이터가 다르다. 두 가지를 출력 문장으로 구분해둔다.

In [13]:
print('검증 지표는 4장에서 X_val과 y_val로 계산한다.')
print('제출 파일에는 reg_model.predict(reg_test_X)의 test 예측값을 저장한다.')

검증 지표는 4장에서 X_val과 y_val로 계산한다.
제출 파일에는 reg_model.predict(reg_test_X)의 test 예측값을 저장한다.


#### 2. 전체 train 학습과 회귀 예측값 저장

전체 train으로 학습해 회귀 예측값을 저장하고 결과를 점검해본다.

In [14]:
reg_model = LinearRegression()
reg_model.fit(reg_X, reg_y)
reg_pred = reg_model.predict(reg_test_X)
reg_result = reg_submit.copy()
reg_result['pred'] = reg_pred
reg_result.to_csv(
    'result_regression.csv',
    index=False
)
reg_check = pd.read_csv('result_regression.csv')
print(reg_check.shape)
print(reg_check.head().round(3))
print(reg_check.isna().sum().sum())
print(
    round(reg_check['pred'].min(), 3),
    round(reg_check['pred'].max(), 3)
)
print(reg_check['pred'].between(0, 1).all())

(280, 1)
     pred
0  62.717
1  61.246
2  47.858
3  42.685
4  42.567
0
24.943 86.234
False


## 07 제출 파일 읽어서 검증하기

### 1 구조 검사

#### 1. 기본 구조 검사

저장된 result.csv를 다시 읽어 크기, 컬럼, 앞부분을 확인해본다.

In [15]:
binary_check = pd.read_csv('result.csv')
print(binary_check.shape)
print(binary_check.columns.tolist())
print(binary_check.head())

(240, 1)
['pred']
   pred
0  0.27
1  0.32
2  0.55
3  0.25
4  0.13


#### 2. 행 수와 컬럼 구조 비교

행 수와 컬럼 구조가 test·제출 예시와 일치하는지 비교해본다.

In [16]:
print(len(binary_check) == len(binary_test))
print(
    binary_check.columns.tolist()
     == binary_submit.columns.tolist())

True
True


### 2 값 검사

#### 1. 결측치와 인덱스 컬럼 확인

결측치와 불필요한 인덱스 컬럼이 없는지 확인해본다.

In [17]:
print(binary_check.isna().sum().sum())
print(any(str(col).startswith('Unnamed')
        for col in binary_check.columns))

0
False


#### 2. 문제 유형별 값 검사

## 08 제출 전 결과 확인

### 1 제출 점검표와 최종 확인

#### 1. 공통 제출 점검표

#### 2. 문제 유형별 pred 최종 확인

### 2 제출 직전 코드와 점검 순서

#### 1. 제출 직전 고정 코드

제출 직전에 그대로 실행할 수 있는 점검 코드는 다음과 같다.

In [18]:
check = pd.read_csv('result.csv')
print('shape: ', check.shape)
print('columns: ', check.columns.tolist())
print('row_match: ', len(check) == len(binary_test))
print(
    'column_match: ',
    check.columns.tolist()
     == binary_submit.columns.tolist())
print('missing: ', check.isna().sum().sum())
print(
    'index_column: ',
    any(
        str(col).startswith('Unnamed')
        for col in check.columns))
print('pred_range: ', round(check['pred'].min(), 2), round(check['pred'].max(), 2))
print('between_0_1: ', check['pred'].between(0, 1).all())
print(check.head())

shape:  (240, 1)
columns:  ['pred']
row_match:  True
column_match:  True
missing:  0
index_column:  False
pred_range:  0.08 0.94
between_0_1:  True
   pred
0  0.27
1  0.32
2  0.55
3  0.25
4  0.13


#### 2. 결과가 잘못되었을 때 점검 순서